# PrizeAtlas: Nobel laureates with their OpenAlex ids

**Source.** [PrizeAtlas](https://prizeatlas.org/es/nobel-prize/) publishes one static page per Nobel award in physics,
chemistry and physiology or medicine (1901–2025), e.g.
[Susumu Kitagawa, Química 2025](https://prizeatlas.org/es/nobel-prize/quimica/2025/susumu-kitagawa/). Each page carries the
motivation, the affiliation at the time of the prize, the co-laureates, birth / death facts and the identifiers **OpenAlex
author id**, ORCID, Wikidata and ROR. Data are CC BY-SA 4.0; `robots.txt` allows crawling (`Allow: /`).

**What the notebook does.**

1. walks the site the way a reader does: the Nobel index → each category page → each prize-year page → each laureate page;
2. parses every laureate page (the facts block, the JSON-LD `Person` record, affiliations, co-laureates);
3. checks the crawl against the site's sitemap and its own `awards.csv` download (ids and extra columns such as the
   institution's OpenAlex id);
4. links the records to the laureates of the SciSciNet / Li et al. (2019) files (`Data/Li2019/`) by field, prize year and
   name, and compares the PrizeAtlas OpenAlex id with the author bridge of `nobel_laureate_papers.ipynb`;
5. writes everything to `Nobel Prize/Data/prizeatlas/`.

**Politeness and reruns.** One request per `DELAY_S` seconds (default 1 s, about 1,000 pages ≈ 20 minutes), retries with
back-off on 429 / 5xx, an identifying User-Agent. Every page is kept under `Data/prizeatlas/html/` (the site's folder layout),
so a rerun parses the saved pages without touching the site; `PA_REFRESH=1` downloads them again.

**How to run.** Needs internet access, so run it on the login node (it uses a few hundred MB), not on a compute node. Kernel
`Python (Curvature)`. `PA_LIMIT=20` parses only the first 20 laureate pages (quick test; the index pages are still read).

## 0. Parameters

In [ ]:
import os
from pathlib import Path

BASE_URL = 'https://prizeatlas.org/es/nobel-prize/'     # Spanish Nobel index (categories -> years -> laureates)
SITE = 'https://prizeatlas.org'
NP = Path('/project/jevans/Dawoon/Nobel Prize')
OUT_DIR = NP / 'Data' / 'prizeatlas'
PAGE_CACHE = OUT_DIR / 'html'                           # raw pages, mirroring the site's folders
DELAY_S = float(os.environ.get('PA_DELAY_S', '1.0'))    # pause between live requests
TIMEOUT_S, MAX_RETRIES = 30, 4
REFRESH = os.environ.get('PA_REFRESH', '0') == '1'      # download pages that are already saved again
LIMIT = int(os.environ.get('PA_LIMIT', '0'))            # > 0: only the first LIMIT laureate pages (quick test)
FETCH_AWARDS_CSV = True                                 # the site's awards.csv, for the cross-check and extra columns
USER_AGENT = 'PrizeAtlasResearchCrawler/1.0 (academic research on Nobel laureates; python-requests)'
LI_DIR = NP / 'Data' / 'Li2019'                                          # SciSciNet / Li et al. (2019) laureate records
BRIDGE_CSV = NP / 'output' / 'nobel_laureates' / 'laureate_author_map.csv'  # author bridge of nobel_laureate_papers.ipynb
print(dict(BASE_URL=BASE_URL, OUT_DIR=str(OUT_DIR), DELAY_S=DELAY_S, REFRESH=REFRESH, LIMIT=LIMIT))

### Setup: polite fetcher with a page cache

In [ ]:
import re, json, time, datetime, difflib, unicodedata, urllib.robotparser
from urllib.parse import urljoin, urlparse
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_colwidth', 60)
PAGE_CACHE.mkdir(parents=True, exist_ok=True)
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': USER_AGENT, 'Accept-Language': 'es'})
ROBOTS = urllib.robotparser.RobotFileParser(urljoin(SITE, '/robots.txt'))
ROBOTS.read()
assert ROBOTS.can_fetch(USER_AGENT, BASE_URL), 'robots.txt disallows the Nobel pages'
LOG, WRITTEN = [], []
_last_request = [0.0]
T_START = time.time()


def cache_path(url):
    # https://prizeatlas.org/es/nobel-prize/quimica/2025/x/ -> html/es/nobel-prize/quimica/2025/x/index.html
    path = urlparse(url).path.strip('/')
    return PAGE_CACHE / path / 'index.html' if path else PAGE_CACHE / 'index.html'


def fetch(url, binary_name=None):
    # the page text, from the cache or the site (None when the site answers with a non-retryable status)
    path = OUT_DIR / binary_name if binary_name else cache_path(url)
    if path.exists() and not REFRESH:
        LOG.append({'url': url, 'status': 200, 'bytes': path.stat().st_size, 'from_cache': True, 'error': None})
        return path.read_text(encoding='utf-8')
    err = None
    for attempt in range(MAX_RETRIES):
        wait = DELAY_S - (time.time() - _last_request[0])
        if wait > 0:
            time.sleep(wait)
        _last_request[0] = time.time()
        try:
            r = SESSION.get(url, timeout=TIMEOUT_S)
        except requests.RequestException as e:
            err = repr(e); time.sleep(DELAY_S * 2 ** attempt); continue
        if r.status_code == 200:
            r.encoding = 'utf-8'
            path.parent.mkdir(parents=True, exist_ok=True)
            tmp = path.with_name(path.name + '.tmp')
            tmp.write_text(r.text, encoding='utf-8'); os.replace(tmp, path)
            LOG.append({'url': url, 'status': 200, 'bytes': len(r.content), 'from_cache': False, 'error': None})
            return r.text
        err = f'HTTP {r.status_code}'
        if r.status_code not in (429, 500, 502, 503, 504):
            break
        retry_after = r.headers.get('Retry-After', '')
        time.sleep(float(retry_after) if retry_after.isdigit() else 5 * 2 ** attempt)
    LOG.append({'url': url, 'status': err, 'bytes': 0, 'from_cache': False, 'error': err})
    print('FAILED', url, err)
    return None


def soup(html):
    return BeautifulSoup(html, 'html.parser')


def text(node):
    return re.sub(r'\s+', ' ', node.get_text(' ', strip=True)).strip() if node is not None else None


def last_segment(href):
    return href.rstrip('/').rsplit('/', 1)[-1]


def save_table(df, name):
    # csv (+ parquet for the main tables), written atomically
    for ext in ('csv', 'parquet') if name in ('prizeatlas_nobel_laureates',) else ('csv',):
        p = OUT_DIR / f'{name}.{ext}'
        tmp = p.with_name(p.name + '.tmp')
        df.to_csv(tmp, index=False) if ext == 'csv' else df.to_parquet(tmp, index=False)
        os.replace(tmp, p); WRITTEN.append(p)
        print(f'-> {p.relative_to(NP)}  ({len(df):,} rows)')

print('robots.txt allows', BASE_URL)

## 1. Index → categories → prize years → laureate pages

The Nobel index lists the categories (`Química`, `Medicina`, `Física`); each category page lists its prize years
(`a.category-year`); each year page lists the awards of that year (`ol.recipient-list`: one item per motivation, the
laureates who share it as links). The frontier keeps the category, year, motivation group and link of every laureate page.

In [ ]:
index = soup(fetch(BASE_URL))
CATEGORIES = []
for a in index.select('ul.route-list a[href]'):
    if re.fullmatch(r'[a-z][a-z-]*/', a['href']):                  # 'quimica/' (year links are digits)
        CATEGORIES.append({'category_slug': a['href'].strip('/'), 'category': text(a), 'url': urljoin(BASE_URL, a['href'])})
print('categories:', [c['category'] for c in CATEGORIES])
assert CATEGORIES, 'no category links on the index page (layout changed?)'

rows = []
for cat in CATEGORIES:
    cpage = soup(fetch(cat['url']))
    years = [(int(text(a)), urljoin(cat['url'], a['href'])) for a in cpage.select('a.category-year[href]')]
    print(f"{cat['category']}: {len(years)} prize years ({min(y for y, _ in years)}–{max(y for y, _ in years)})")
    for year, yurl in years:
        ypage = soup(fetch(yurl))
        if ypage is None:
            continue
        for g, li in enumerate(ypage.select('ol.recipient-list > li'), start=1):
            motivation = next((text(p) for p in li.find_all('p', recursive=False) if not p.get('class')), None)
            names = li.select('h3 a[href]')
            for a in names:
                rows.append({'category_slug': cat['category_slug'], 'category': cat['category'], 'year': year, 'award_group': g,
                             'group_size': len(names), 'year_page_motivation': motivation, 'slug': last_segment(a['href']),
                             'link_name': text(a), 'url': urljoin(yurl, a['href'])})
FRONTIER = pd.DataFrame(rows).drop_duplicates('url').reset_index(drop=True)
print(f'{len(FRONTIER):,} laureate pages from {FRONTIER[["category_slug", "year"]].drop_duplicates().shape[0]} category-years')
display(FRONTIER.groupby('category').agg(pages=('url', 'size'), first_year=('year', 'min'), last_year=('year', 'max')))

# the sitemap is a second, independent list of the same pages
sm = fetch(urljoin(SITE, '/sitemap.xml'))
sitemap = {u for u in re.findall(r'<loc>(.*?)</loc>', sm or '') if re.fullmatch(re.escape(BASE_URL) + r'[a-z-]+/\d{4}/[^/]+/', u)}
only_sitemap, only_crawl = sorted(sitemap - set(FRONTIER.url)), sorted(set(FRONTIER.url) - sitemap)
print(f'sitemap: {len(sitemap):,} laureate pages; only in the sitemap: {len(only_sitemap)}; only in the crawl: {len(only_crawl)}')
if only_sitemap:
    print('  added from the sitemap:', only_sitemap[:10])
    extra = pd.DataFrame({'url': only_sitemap})
    parts = extra.url.str.extract(re.escape(BASE_URL) + r'(?P<category_slug>[a-z-]+)/(?P<year>\d{4})/(?P<slug>[^/]+)/')
    extra = pd.concat([extra, parts], axis=1).assign(year=lambda x: x.year.astype(int), found_via='sitemap')
    FRONTIER = pd.concat([FRONTIER.assign(found_via='index'), extra], ignore_index=True)
else:
    FRONTIER['found_via'] = 'index'
if LIMIT:
    FRONTIER = FRONTIER.head(LIMIT)
    print(f'LIMIT={LIMIT}: parsing {len(FRONTIER)} pages')

## 2. Laureate pages

Per page: name (`h1`), motivation (`p.citation`), subject badges, the facts block (`aside.facts dl`: prize share, birth and
death date / city / country, citizenship, **ORCID, Wikidata (`WDATA`), OpenAlex, ROR**), the JSON-LD `Person` record,
the affiliation(s) at the time of the prize (institution, the sub-unit when the page names one, place, PrizeAtlas institution
slug), the co-laureates, the English URL and the person page. Fact labels are Spanish on these pages and are mapped to English
column names; an unknown label is kept as `fact_<label>` and reported.

In [ ]:
FACT_LABELS = {'reparto de premios': 'prize_share_text', 'nacimiento': 'birth_date', 'ciudad de nacimiento': 'birth_city',
               'país de nacimiento': 'birth_country', 'ciudadanía': 'citizenship', 'muerte': 'death_date',
               'ciudad de la muerte': 'death_city', 'país de la muerte': 'death_country',
               'orcid': 'orcid', 'wdata': 'wikidata_qid', 'openalex': 'openalex_author_id', 'ror': 'ror'}
UNKNOWN_LABELS = {}


def slug_label(s):
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()
    return 'fact_' + re.sub(r'[^a-z0-9]+', '_', s).strip('_')


def parse_facts(page):
    out = {}
    for div in page.select('aside.facts dl > div'):
        dt, dd = div.find('dt'), div.find('dd')
        if dt is None or dd is None:
            continue
        label = text(dt)
        key = FACT_LABELS.get(label.lower())
        if key is None:
            key = slug_label(label); UNKNOWN_LABELS[label] = UNKNOWN_LABELS.get(label, 0) + 1
        links = [text(a) for a in dd.select('a')]
        out[key] = '; '.join(links) if len(links) > 1 else text(dd)
    return out


def parse_affiliations(page):
    affs = []
    for i, div in enumerate(page.select('section.winner-affiliations > div'), start=1):
        inst, slug, place, plain = None, None, None, []
        for p in div.find_all('p'):
            a = p.find('a', href=re.compile(r'/instituciones/'))
            if 'place' in (p.get('class') or []):
                place = text(p)
            elif a is not None and inst is None:
                inst, slug = text(a), last_segment(a['href'])
            else:
                plain.append(text(p))
        if inst is None and plain:                  # institution without a PrizeAtlas page
            inst, plain = plain[0], plain[1:]
        affs.append({'affiliation_order': i, 'institution': inst, 'institution_slug': slug,
                     'sub_unit': '; '.join(plain) or None, 'place': place})
    return affs


def json_ld_person(page):
    for s in page.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(s.string or '')
        except json.JSONDecodeError:
            continue
        for node in data.get('@graph', [data]):
            if node.get('@type') == 'Person':
                return node
    return {}


def parse_laureate(html, url):
    page = soup(html)
    art = page.select_one('article.winner-article')
    rec = {'url': url, 'name': text(art.select_one('h1')) if art else text(page.find('h1')),
           'motivation': text(page.select_one('p.citation')),
           'subjects': '; '.join(text(a) for a in page.select('p.subject-badges a.subject-badge')) or None}
    rec.update(parse_facts(page))
    # identifiers: the facts block first, the external links as a fallback
    for key, pat in [('openalex_author_id', r'openalex\.org/authors/(A\d+)'), ('orcid', r'orcid\.org/([0-9X-]{19})'),
                     ('wikidata_qid', r'wikidata\.org/wiki/(Q\d+)'), ('ror', r'ror\.org/([0-9a-z]{9})')]:
        found = list(dict.fromkeys(m.group(1) for a in page.select('a[href]') if (m := re.search(pat, a['href']))))
        if not rec.get(key) and found:
            rec[key] = '; '.join(found)
        rec[f'n_{key}_links'] = len(found)
    m = re.search(r'(\d+)\s*/\s*(\d+)', rec.get('prize_share_text') or '')
    rec['prize_share'] = f'{m.group(1)}/{m.group(2)}' if m else None
    rec['prize_share_value'] = int(m.group(1)) / int(m.group(2)) if m else np.nan
    person = json_ld_person(page)
    rec['jsonld_birth_place'] = (person.get('birthPlace') or {}).get('name') if isinstance(person.get('birthPlace'), dict) else None
    aff_ld = person.get('affiliation')
    aff_ld = aff_ld if isinstance(aff_ld, list) else [aff_ld] if aff_ld else []
    rec['jsonld_affiliations'] = '; '.join(a.get('name', '') for a in aff_ld if isinstance(a, dict)) or None
    rec['jsonld_award'] = person.get('award')
    rec['jsonld_same_as'] = '; '.join(person['sameAs']) if isinstance(person.get('sameAs'), list) else person.get('sameAs')
    affs = parse_affiliations(page)
    rec['n_affiliations'] = len(affs)
    for k in ('institution', 'institution_slug', 'sub_unit', 'place'):
        rec[f'affiliation_{k}'] = '; '.join(a[k] or '' for a in affs) if affs else None
    co = page.select('ul.co-laureates a[href]')
    rec['co_laureates'] = '; '.join(text(a) for a in co) or None
    rec['co_laureate_slugs'] = '; '.join(last_segment(a['href']) for a in co) or None
    rec['n_co_laureates'] = len(co)
    en = page.select_one('link[rel="alternate"][hreflang="en"]')
    rec['url_en'] = en['href'] if en else None
    pp = page.select_one('nav.breadcrumbs a[href*="/personas/"]')
    rec['person_slug'] = last_segment(pp['href']) if pp else None
    built = page.select_one('p.built')
    rec['site_updated'] = (re.search(r'\d{4}-\d{2}-\d{2}', text(built) or '') or [None])[0] if built else None
    return rec, affs


recs, aff_rows, failed = [], [], []
t0 = time.time()
for i, r in enumerate(FRONTIER.itertuples(index=False), start=1):
    html = fetch(r.url)
    if html is None:
        failed.append(r.url); continue
    try:
        rec, affs = parse_laureate(html, r.url)
    except Exception as e:                         # keep going; the page is listed as failed
        print('PARSE ERROR', r.url, repr(e)); failed.append(r.url); continue
    recs.append({**r._asdict(), **rec})
    aff_rows += [{'url': r.url, 'category_slug': r.category_slug, 'year': r.year, 'slug': r.slug, **a} for a in affs]
    if i % 100 == 0:
        print(f'{i:,} / {len(FRONTIER):,} pages ({time.time() - t0:.0f}s)')
print(f'parsed {len(recs):,} pages, failed {len(failed)} in {time.time() - t0:.0f}s')
if UNKNOWN_LABELS:
    print('fact labels without a mapping (kept as fact_*):', UNKNOWN_LABELS)

W = pd.DataFrame(recs)
W['category_en'] = W.url_en.str.extract(r'/nobel-prize/([a-z-]+)/\d{4}/')[0].str.capitalize()
W['wikidata_qid'] = W.wikidata_qid.fillna(W.jsonld_same_as.str.extract(r'(Q\d+)')[0])
W['birth_year'] = pd.to_numeric(W.birth_date.str.slice(0, 4), errors='coerce').astype('Int64')
W['age_at_prize'] = (W.year - W.birth_year).astype('Int64')
W['person_key'] = W.wikidata_qid.fillna('slug:' + W.person_slug.fillna(W.slug))
AFF = pd.DataFrame(aff_rows)
lead = ['category_en', 'category', 'category_slug', 'year', 'name', 'openalex_author_id', 'orcid', 'wikidata_qid', 'ror',
        'motivation', 'prize_share', 'prize_share_value', 'affiliation_institution', 'affiliation_sub_unit', 'affiliation_place',
        'birth_date', 'birth_city', 'birth_country', 'citizenship', 'death_date', 'death_city', 'death_country', 'age_at_prize',
        'co_laureates', 'n_co_laureates', 'award_group', 'group_size', 'slug', 'person_slug', 'person_key', 'url', 'url_en']
W = W[[c for c in lead if c in W] + [c for c in W if c not in lead]].sort_values(['category_en', 'year', 'award_group', 'name'], kind='mergesort')
display(W.head(8))

## 3. Checks

Coverage of every field, pages without an OpenAlex id, pages whose identifier links disagree with the facts block, people with
several prizes (Curie, Bardeen, Sanger, Sharpless) and whether they carry one OpenAlex id, and the year pages' motivation
against the laureate page's.

In [ ]:
cov = W.notna().mean().rename('share_filled').to_frame()
cov['filled'] = W.notna().sum()
display(cov.loc[[c for c in lead if c in W]].T.round(3))
dec = W.assign(decade=W.year // 10 * 10).groupby(['category_en', 'decade']).agg(
    pages=('url', 'size'), openalex=('openalex_author_id', lambda s: s.notna().mean()), orcid=('orcid', lambda s: s.notna().mean()),
    ror=('ror', lambda s: s.notna().mean()))
print('share of pages with each id, by category and decade:'); display(dec.unstack(0).round(2))

no_oa = W[W.openalex_author_id.isna()][['category_en', 'year', 'name', 'url']]
print(f'pages without an OpenAlex id: {len(no_oa)}'); display(no_oa)
multi = W[W.n_openalex_author_id_links.fillna(0) > 1][['category_en', 'year', 'name', 'openalex_author_id']]
print(f'pages linking more than one OpenAlex author: {len(multi)}'); display(multi.head(10))
bad = W[W.openalex_author_id.notna() & ~W.openalex_author_id.str.fullmatch(r'A\d+(; A\d+)*', na=False)]
print(f'malformed OpenAlex ids: {len(bad)}')
mot = W.dropna(subset=['motivation', 'year_page_motivation'])
print(f'motivation on the laureate page equals the year page: {(mot.motivation == mot.year_page_motivation).mean():.3f}')

P = (W.groupby('person_key')
      .agg(name=('name', 'first'), n_prizes=('url', 'size'), categories=('category_en', lambda s: '; '.join(dict.fromkeys(s))),
           prize_years=('year', lambda s: '; '.join(map(str, sorted(s)))), openalex_author_ids=('openalex_author_id', lambda s: '; '.join(dict.fromkeys(s.dropna()))),
           orcid=('orcid', 'first'), wikidata_qid=('wikidata_qid', 'first'), birth_date=('birth_date', 'first'), death_date=('death_date', 'first'),
           person_slug=('person_slug', 'first'))
      .reset_index().sort_values(['n_prizes', 'name'], ascending=[False, True]))
P['n_openalex_ids'] = P.openalex_author_ids.apply(lambda s: len(s.split('; ')) if s else 0)
print(f'{len(P):,} people for {len(W):,} awards; several prizes:'); display(P[P.n_prizes > 1])
print('people with more than one OpenAlex id across their pages:', int((P.n_openalex_ids > 1).sum()))

## 4. Cross-check with the site's `awards.csv`

PrizeAtlas also offers the whole database as one CSV (`/awards.csv`, all prizes). Its Nobel rows are matched to the crawled
pages on (English category, year, laureate Wikidata id), the identifiers compared, and the columns the pages do not show
(`award_record_id`, `sex`, `laureate_type`, the affiliation's Wikidata id and the **institution's OpenAlex id**) added with
the prefix `csv_`.

In [ ]:
if FETCH_AWARDS_CSV:
    fetch(urljoin(SITE, '/awards.csv'), binary_name='awards.csv')
    AW = pd.read_csv(OUT_DIR / 'awards.csv', dtype=str)
    WRITTEN.append(OUT_DIR / 'awards.csv')
    AWN = AW[AW.prize.fillna('').str.fullmatch(r'Nobel Prize.*') | AW.award_record_id.fillna('').str.startswith('nobel-')].copy()
    AWN['year'] = AWN.year.astype(int)
    print(f'awards.csv: {len(AW):,} rows, {len(AWN):,} Nobel rows; by category {AWN.category.value_counts().to_dict()}')
    keep = {'award_record_id': 'csv_award_record_id', 'sex': 'csv_sex', 'laureate_type': 'csv_laureate_type',
            'affiliation_wikidata_qid': 'csv_affiliation_wikidata_qid', 'institution_openalex_id': 'csv_institution_openalex_id',
            'author_openalex_id': 'csv_author_openalex_id', 'orc_id': 'csv_orcid', 'affiliate_ror': 'csv_ror', 'full_name': 'csv_full_name',
            'affiliation_name': 'csv_affiliation_name', 'affiliation_sub_name': 'csv_affiliation_sub_name', 'award_wikidata_qid': 'csv_award_wikidata_qid'}
    X = AWN.rename(columns={'category': 'category_en', 'laureate_wikidata_qid': 'wikidata_qid'})[['category_en', 'year', 'wikidata_qid', *keep]].rename(columns=keep)
    dup = X.duplicated(['category_en', 'year', 'wikidata_qid'], keep=False)
    print('awards.csv rows sharing (category, year, Wikidata id):', int(dup.sum()))
    W = W.drop(columns=[c for c in W if c.startswith('csv_')]).merge(X[~dup], on=['category_en', 'year', 'wikidata_qid'], how='left', indicator='csv_match')
    W['csv_match'] = W.csv_match.eq('both')
    print(f"pages matched to an awards.csv row: {int(W.csv_match.sum())} of {len(W)}")
    agree = {}
    for col, other in [('openalex_author_id', 'csv_author_openalex_id'), ('orcid', 'csv_orcid'), ('ror', 'csv_ror')]:
        both = W[W.csv_match & W[col].notna() & W[other].notna()]
        agree[col] = {'both_present': len(both), 'equal': int((both[col] == both[other]).sum()),
                      'only_page': int((W.csv_match & W[col].notna() & W[other].isna()).sum()),
                      'only_csv': int((W.csv_match & W[col].isna() & W[other].notna()).sum())}
    display(pd.DataFrame(agree).T)
    diff = W[W.csv_match & W.openalex_author_id.notna() & W.csv_author_openalex_id.notna() & (W.openalex_author_id != W.csv_author_openalex_id)]
    if len(diff):
        print('OpenAlex id differs between the page and awards.csv:'); display(diff[['category_en', 'year', 'name', 'openalex_author_id', 'csv_author_openalex_id']])
    print('unmatched pages:'); display(W[~W.csv_match][['category_en', 'year', 'name', 'wikidata_qid', 'url']].head(20))

## 5. Link to the SciSciNet / Li et al. (2019) laureates

The laureates of `Data/Li2019/` (1902–2016; the `LaureateID` of `SciSciNet_Link_NobelLaureates.tsv`) are matched within
field × prize year on the family name: Li et al. names come as `last, initials`, MAG-style truncated `baltimor.d`, or full
names. Family names are matched as written, with hyphens / spaces dropped (`barresinoussi`), truncated (`baltimor`), in reversed
order (`thomas ed`) or by spelling similarity; several laureates with one family name (the Braggs) are told apart by the
given-name initials, honorifics such as *Sir* left out. A page claimed by two Li laureates stays unlinked. The PrizeAtlas
OpenAlex id is then compared with the author bridge of `nobel_laureate_papers.ipynb` (`laureate_author_map.csv`: the most
frequent matching author id and the fragment ids), when that file exists.

In [ ]:
def fold(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z ]+', ' ', s).split()


HONORIFICS = {'sir', 'lord', 'dame', 'lady', 'baron'}
PARTICLES = {'von', 'van', 'der', 'den', 'de', 'du', 'da', 'di', 'la', 'le', 'af', 'y'}


def li_parts(name):
    # 'thouless, dj' -> ('thouless', 'dj'); 'baltimor.d' -> ('baltimor', 'd'); 'Albert Einstein' -> ('einstein', 'a');
    # 'Novoselov, KOSTYAS' -> ('novoselov', 'k'): a single given token of up to three letters is read as initials
    name = str(name).strip()
    if ',' in name:
        last, first = name.split(',', 1)
    elif re.fullmatch(r'[^\s.]+\.[a-z]+', name):
        last, first = name.split('.', 1)
    else:
        toks = name.split(); last, first = toks[-1], ' '.join(toks[:-1])
    lt, ft = fold(last), fold(first)
    initials = ft[0] if len(ft) == 1 and len(ft[0]) <= 3 else ''.join(t[0] for t in ft)
    return ' '.join(lt), initials


LI = []
for f in sorted(LI_DIR.glob('* publication record.tab')):
    field = f.name.split()[0]
    t = pd.read_csv(f, usecols=['Laureate ID', 'Laureate name', 'Prize year'], dtype=str, encoding='utf-8',   # Physics has stray non-UTF-8 bytes
                    encoding_errors='replace').drop_duplicates('Laureate ID')
    LI.append(t.assign(field=field))
LI = pd.concat(LI, ignore_index=True).rename(columns={'Laureate ID': 'LaureateID', 'Laureate name': 'li_name', 'Prize year': 'prize_year'})
LI['LaureateID'] = LI.LaureateID.astype(int); LI['prize_year'] = LI.prize_year.astype(int)
LI[['li_last', 'li_initials']] = LI.li_name.apply(lambda s: pd.Series(li_parts(s)))
print(f'Li et al. laureates: {len(LI):,} ({LI.field.value_counts().to_dict()})')


def name_rule(li_last, pa_name):
    # the rule by which a Li family name matches a PrizeAtlas name, or None
    toks = fold(pa_name)
    if not li_last or not toks:
        return None
    if f' {li_last} ' in f" {' '.join(toks)} ":                               # family name, incl. compound 'de gennes'
        return 'family name'
    joined = li_last.replace(' ', '')
    if len(joined) >= 6 and joined in ''.join(toks):                         # hyphens / spaces dropped: 'barresinoussi', 'vandermeer'
        return 'family name (joined)'
    if len(li_last) >= 7 and any(t.startswith(li_last) for t in toks[1:]):   # truncated MAG form 'baltimor'
        return 'truncated family name'
    return None


def given_initials(pa_name, li_last):
    # initials of the given names: honorifics, particles and the family name left out ('Sir William Henry Bragg' -> 'wh')
    fam = set(li_last.split())
    return ''.join(t[0] for t in fold(pa_name) if t not in HONORIFICS and t not in PARTICLES and t not in fam)


links = []
Wf = W.rename(columns={'category_en': 'field'})
for r in LI.itertuples(index=False):
    cand = Wf[(Wf.field == r.field) & (Wf.year == r.prize_year)]
    rules = cand.name.apply(lambda n: name_rule(r.li_last, n))
    hits, rule = cand[rules.notna()], (rules.dropna().iloc[0] if rules.notna().any() else None)
    if hits.empty:                 # reversed order ('thomas ed', 'von euler us'): a longer token of the Li name as the family name
        for tok in [t for t in fold(r.li_name) if len(t) >= 4 and t not in PARTICLES]:
            h = cand[cand.name.apply(lambda n: tok in fold(n)[1:])]
            if len(h):
                hits, rule = h, 'family name (reversed order)'; break
    if hits.empty:                 # spelling variants: closest family name in the field-year
        best = difflib.get_close_matches(r.li_last, [' '.join(fold(n)[-1:]) for n in cand.name], n=1, cutoff=0.8)
        if best:
            hits, rule = cand[cand.name.apply(lambda n: ' '.join(fold(n)[-1:]) == best[0])], 'fuzzy family name'
    if len(hits) > 1 and r.li_initials:  # several laureates with the family name (the Braggs): the given-name initials decide
        h = hits[hits.name.apply(lambda n: given_initials(n, r.li_last).startswith(r.li_initials))]
        if len(h):
            hits, rule = h, f'{rule} + initials'
    links.append({'LaureateID': r.LaureateID, 'li_name': r.li_name, 'field': r.field, 'prize_year': r.prize_year,
                  'n_candidates': len(hits), 'match_rule': rule if len(hits) == 1 else ('ambiguous' if len(hits) else 'no match'),
                  'url': hits.url.iloc[0] if len(hits) == 1 else None})
LINK = pd.DataFrame(links)
claims = LINK.url.value_counts()
shared = LINK.url.isin(claims[claims > 1].index)
if shared.any():                   # two Li laureates on one page (e.g. both 1931 chemists are 'carl bosch' in Li et al.): leave both unlinked
    print('pages claimed by several Li laureates (left unlinked):'); display(LINK[shared])
    LINK.loc[shared, 'match_rule'] = 'ambiguous (page claimed by several Li laureates)'
    LINK.loc[shared, 'url'] = None
LINK = LINK.merge(W[['url', 'name', 'openalex_author_id', 'orcid', 'wikidata_qid']], on='url', how='left')
print('match rules:', LINK.match_rule.value_counts().to_dict())
display(LINK[LINK.match_rule.str.startswith(('no match', 'ambiguous'))])

if BRIDGE_CSV.exists():
    B = pd.read_csv(BRIDGE_CSV, usecols=['LaureateID', 'primary_author_id', 'fragment_ids'], dtype={'fragment_ids': str})
    LINK = LINK.merge(B, on='LaureateID', how='left')
    has = LINK.openalex_author_id.notna() & LINK.primary_author_id.notna()
    LINK['bridge_agrees'] = np.where(has, LINK.openalex_author_id == LINK.primary_author_id, pd.NA)
    LINK['in_bridge_fragments'] = [bool(o) and isinstance(fr, str) and o in fr.split(';') for o, fr in zip(LINK.openalex_author_id, LINK.fragment_ids)]
    print(f'laureates with both ids: {int(has.sum())}; PrizeAtlas id = bridge primary id: {LINK.bridge_agrees.eq(True).sum()} '
          f'({LINK.bridge_agrees.eq(True).sum() / max(1, has.sum()):.1%}); among the bridge fragments: {int(LINK.in_bridge_fragments.sum())}')
    display(LINK[LINK.bridge_agrees.eq(False)][['field', 'prize_year', 'li_name', 'name', 'openalex_author_id', 'primary_author_id', 'fragment_ids']].head(15))
else:
    print(f'{BRIDGE_CSV} not found: run nobel_laureate_papers.ipynb for the bridge comparison')

## 6. Save

In [ ]:
save_table(W, 'prizeatlas_nobel_laureates')
save_table(AFF, 'prizeatlas_nobel_affiliations')
save_table(P, 'prizeatlas_nobel_people')
save_table(LINK, 'prizeatlas_sciscinet_laureate_link')
log = pd.DataFrame(LOG)
save_table(log, 'crawl_log')

source = f'''# PrizeAtlas Nobel laureates

Crawled by `Nobel Prize/notebook/prizeatlas_crawl.ipynb` on {datetime.date.today().isoformat()} from {BASE_URL}
(index -> category -> prize year -> laureate page; site build date {W.site_updated.dropna().max() if 'site_updated' in W else 'n/a'}).
Data: PrizeAtlas, CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/); code https://github.com/antb123/prizeatlas.

| file | content |
|---|---|
| `prizeatlas_nobel_laureates.csv` / `.parquet` | one row per Nobel award page ({len(W):,}): ids (`openalex_author_id`, `orcid`, `wikidata_qid`, `ror`), motivation, prize share, affiliation, birth / death, co-laureates, URLs; `csv_*` = columns of the site's awards.csv |
| `prizeatlas_nobel_affiliations.csv` | one row per page × affiliation ({len(AFF):,}) |
| `prizeatlas_nobel_people.csv` | one row per person ({len(P):,}); `person_key` = Wikidata id |
| `prizeatlas_sciscinet_laureate_link.csv` | Li et al. (2019) / SciSciNet `LaureateID` -> PrizeAtlas page and OpenAlex id ({int(LINK.url.notna().sum()):,} of {len(LINK):,} matched) |
| `awards.csv` | the site's own download (all prizes), as fetched |
| `crawl_log.csv`, `manifest.json` | requests and run summary |
| `html/` | the raw pages (the site's folder layout); a rerun parses these |

Categories are the Spanish slugs of the crawled pages (`quimica`, `medicina`, `fisica`); `category_en` comes from the English URL.
Place names are the Spanish page text; the `csv_*` columns and `awards.csv` are in English.
'''
p = OUT_DIR / 'SOURCE.md'; p.write_text(source, encoding='utf-8'); WRITTEN.append(p)
manifest = {'finished': datetime.datetime.now().isoformat(timespec='seconds'), 'base_url': BASE_URL, 'delay_s': DELAY_S, 'limit': LIMIT,
            'refresh': REFRESH, 'pages': len(W), 'failed': failed, 'unknown_fact_labels': UNKNOWN_LABELS,
            'requests_live': int((~log.from_cache).sum()) if len(log) else 0, 'requests_cached': int(log.from_cache.sum()) if len(log) else 0,
            'with_openalex': int(W.openalex_author_id.notna().sum()), 'people': len(P),
            'li_matched': int(LINK.url.notna().sum()), 'li_laureates': len(LINK), 'seconds': round(time.time() - T_START, 1),
            'outputs': [str(Path(x).relative_to(NP)) for x in WRITTEN]}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=1, default=str), encoding='utf-8')
print(json.dumps({k: v for k, v in manifest.items() if k != 'outputs'}, indent=1, default=str))